# Fitbit test pull (Google Health API)

**Data source: real** (your own Fitbit). Raw responses are saved to `data/raw/fitbit/` (gitignored).
Outputs are stripped on commit by nbstripout; still, never paste outputs into slides or the writeup.

Question: which data types come back through the course OAuth app, with which fields, and how often?

Setup: `GOOGLE_HEALTH_REFRESH_TOKEN` and `GOOGLE_HEALTH_HELPER_TOKEN` in `.env` (see `.env.example`).

In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "signals").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from signals import google_health as gh  # noqa: E402 (needs ROOT on sys.path)

DAYS_BACK = 35
END = date.today() + timedelta(days=1)  # exclusive, so today is included
START = END - timedelta(days=DAYS_BACK)
DATA_TYPES = list(gh.FILTER_FIELDS)
print(f"Pulling {len(DATA_TYPES)} data types, {START} to {END} (exclusive)")

## 1. What comes back

`forbidden` = the course app's scopes don't cover it. `bad request` = our filter field guess is wrong.
`empty` = allowed, but no data in the window.

In [ ]:
results = gh.probe(DATA_TYPES, START, END, save=True)

print(f"{'data type':38} {'status':24} {'points':>7}")
for r in results:
    print(f"{r['data_type']:38} {r['status']:24} {r['n_points']:>7}")

## 2. Which fields each type has

In [ ]:
for r in results:
    if r["fields"]:
        print(f"\n{r['data_type']}  ->  {r['saved']}")
        for path in r["fields"]:
            print(f"    {path}")

## 3. How often (next step)

Sampling interval and days covered per type are computed once we've seen the timestamp shapes
in the saved JSON.